# Cervical Cancer Cell Classification
#### This notebook tests 5 ML algorithms with the Data_New folder and analyzes which one predicts the best
- Involves splitting into train/validation, normalization, encoding labels, training the ML models, testing, and outputting results
- 6 ML models tested: Logistic Regression, Support Vector Machine, XGBoost, CNN, ResNet50, EfficientNet
- LSTM & GRUs & memory based ML models do not make sense in the context of image classification

In [2]:
import os
from pathlib import Path
from collections import defaultdict
from PIL import Image
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import cv2

#ML model import statements
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import EfficientNetB0

#evaluation metrics import statements
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [3]:
# saving universal variables and lists
valid_ext = ['.bmp']
ROOT_DIR = Path('/Users/anjalisrinivasan/Desktop/projects/cervical_cell_classifier/data_new')

### Splitting the Training set into Train + Validate

In [21]:
# function to separate the files that are labels vs the actual image files
# will then separate the actual image files into train vs validate
def define_path_label(ROOT_DIR, split, valid_ext):
    # should only be done on the training dataset
    data_dir = ROOT_DIR / split
    image_paths = []
    labels = []

    #for every path that is valid, add that image path to the list and its folder's name to labels
    for path in data_dir.rglob("*"):
        if path.suffix.lower() in valid_ext:
            if '.ipynb_checkpoints' in path.parts:
                continue 
            image_paths.append(path)
            labels.append(path.parent.name)
    
    return image_paths, labels

# sanity check to ensure there ar eonly 6 labels and several image paths found & if the format is correct
image_paths, labels = define_path_label(ROOT_DIR, 'Training', valid_ext)
print(f"Number of images found: {len(image_paths)}")
print(f"Number of unique labels found: {len(set(labels))}")
print(f"Example format: {image_paths[0]} -> {labels[0]}")

# actual splitting of training data into tran + validation
# test_size = 0.2 means 20% goes to validation and 80% goes to training
train_paths, valid_paths, train_labels, valid_labels = train_test_split(image_paths, labels, test_size = 0.2, random_state = 42, stratify = labels)

#
print(f"Training set size: {len(train_paths)}")
print(f"Validation set size: {len(valid_paths)}")
#check to see if stratify worked -> if the proportions of images between the categories are similar and preserved
print(f"Training class distribution: {Counter(train_labels)}")
print(f"Validation class distribution:, {Counter(valid_labels)}")

Number of images found: 3549
Number of unique labels found: 5
Example format: /Users/anjalisrinivasan/Desktop/projects/cervical_cell_classifier/data_new/Training/im_Metaplastic/123_02.bmp -> im_Metaplastic
Training set size: 2839
Validation set size: 710
Training class distribution: Counter({'im_Superficial-Intermediate': 585, 'im_Koilocytotic': 580, 'im_Dyskeratotic': 570, 'im_Metaplastic': 554, 'im_Parabasal': 550})
Validation class distribution:, Counter({'im_Superficial-Intermediate': 146, 'im_Koilocytotic': 145, 'im_Dyskeratotic': 143, 'im_Metaplastic': 139, 'im_Parabasal': 137})


### Normalization
#### Each image is pixel encoded from 0-255, but these numbers may be too large for many models, so must normalize it all into one scale
#### The scale decided was 0-1 rescaling; This decision was made purely from training data to avoid data leakage
#### This is only done to the X variable Train, Validate, Test sections -> none of the Y is impacted here because those are all labels

In [ ]:
#function that takes an image, gets its pixel numbers, rescales it to be bewtween 0-1
def image_normalization(path):
    img = Image.open(path).convert('RGB')
    pre_array = np.array(img, dtype = 'float32') #should output an array with 3 things, pixel dimensions and color channels
    normalized_array = pre_array / 255.0
    return normalized_array


#X = the input features that we are USING TO predict the outcomes -> the image paths
X_train = np.array([image_normalization(path) for path in train_paths])
X_valid = np.array([image_normalization(path) for path in valid_paths])

test_paths, test_labels = define_path_label(ROOT_DIR, 'Test', valid_ext)
X_test = np.array([image_normalization(path) for path in test_paths])

# double checks the shape of all 3 that have been normalized to ensure it worked properly
# should result in (num images, pixel width, pixel height, color channel) -> all num images should add up to total calulated earlier
print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)
# print a sample range of the X train to see if it actually rescaled to between 0-1
print(X_train.min(), X_train.max())

(2839, 170, 170, 3)
(710, 170, 170, 3)
(500, 170, 170, 3)
0.050980392156862744 1.0


### Label Encoding
#### Now that the training data has been split and normalized, we must complete label encoding for only Y of the train, validate, and test sections of the labels. Main job of the label encoder is to find unique categories and turn those labels into numbers. These make it easier for the ML model to process and predict outcomes. The mapping of the string to the integer is stored in le and can be printed out for verification.
#### LabelEncode can also decode to revert the numbers into labels when doing real world predictions.

In [26]:
le = LabelEncoder()

# fit_transform learns the data and its associations to the label -> only needs to be learned once for the same group of data or else it will
# relearn from scratch for the validation and test data
Y_train = le.fit_transform(train_labels)
Y_valid = le.transform(valid_labels)
Y_test = le.transform(test_labels)

# sanity check to see the first 10 lines and print out the integer mapping 
print(Y_train[:10]) # should be numbers instead of string labels
print(le.classes_) # should show the classes identified from the labels

[0 2 3 3 3 1 3 4 4 2]
['im_Dyskeratotic' 'im_Koilocytotic' 'im_Metaplastic' 'im_Parabasal'
 'im_Superficial-Intermediate']


In [5]:
##Both ResNet50 and EfficientNetB0 expect 0-255 scaling, so basically revert the scaling back from 0-1 so it can be used.
from tensorflow.keras.applications.resnet50 import preprocess_input

#Resets all the X values for ResNet50 to be back to 0-255 scaling
X_train_resnet = preprocess_input(X_train * 255.0)
X_valid_resnet = preprocess_input(X_valid * 255.0)
X_test_resnet  = preprocess_input(X_test * 255.0)

#Resets all the X values for EfficientNetB0 back to 0-255 scaling
X_train_eff = X_train * 255.0
X_valid_eff = X_valid * 255.0
X_test_eff  = X_test * 255.0

NameError: name 'X_train' is not defined

### Model Fitting
#### All data has been prepared. We will first run 3 models that do not require feature extraction: CNN, ResNet50, and EfficientNetB0.
#### Image classification has typically required feature extraction, where we (the researchers) define the features which the models should base their predictions off of. But these 3 are deeper learning models that take raw pixesl and learn the features themselves.
#### This can be a plus because a lot of cytology images like cancer tissue images might have complex, subtle texture or morphology patterns that are hard to engineer by hand (especially considering that many of these images might not be as high-resolution as the current dataset being used if they come from regions with existing accessibility issues)

In [ ]:
from tensorflow.keras import layers, models

# create the architecture for the CNN model
cnn_model = models.Sequential([
    layers.Input(shape=(170, 170, 3)),
    
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = cnn_model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), epochs=20, batch_size=32)



